# Namespaces `⎕NS`

> The utility of a language as a tool of thought increases with the range of topics it can treat, but decreases with the amount of vocabulary and the complexity of grammatical rules which the user must keep in mind. Economy of notation is therefore important. --_Kenneth E. Iverson_ [@iverson1980notation]

If you come from another programming language, Dyalog's namespaces will appear confusing: they're used both as an associative array (dict) and as a hierarchical code-organisational construct -- you know, an _actual_ namespace. Dyalog describes namespaces like so:

> Namespace is a (class 9) object in Dyalog APL. Namespaces are analogous to nested workspaces.

No, that doesn't mean anything to me either. In fact, the code-organisational aspect of APL namespaces are similar in spirit to those found in [C++](https://en.cppreference.com/w/cpp/language/namespace):

> Namespaces provide a method for preventing name conflicts in large projects. Symbols declared inside a namespace block are placed in a named scope that prevents them from being mistaken for identically-named symbols in other scopes.

except that Dyalog's namespaces don't obey the time-honoured name scoping rules you expect. Sadly. 

Other resources:

* Dyalog docs [Namespaces](https://dyalog.github.io/documentation/21.0/programming-reference-guide/introduction/namespaces/namespaces/) [⎕VSET](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/vset/) [⎕VGET](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/vget/)
* [APL Wiki](https://aplwiki.com/wiki/Namespace)
* [Cultivation](https://xpqz.github.io/cultivations/Namespaces.html)
* [Namespaces in Dyalog APL](https://www.youtube.com/watch?v=eBEwBSO5mBA) video intro

In [8]:
⎕IO ← 0
]box on -s=min
]rows on

Was ON -style=max

Was ON

## 

## Namespaces as dicts

There are many ways to construct namespaces in Dyalog APL. The simplest is perhaps to use APLAN,

In [6]:
⊢ ns ← (
  author: 'Gödel, Kurt'
  title: 'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I'
  booktitle: 'Kurt Gödel: Collected Works, Volume I'
  publisher: 'Oxford University Press'
  year: 1931
  note: 'trans. Jean van Heijenoort, publ. 1986'
)


#.[Namespace]

Namespaces don't display, and APLers see this as a feature (and if you think of the other use of namespaces in APL -- actual namespaces -- you can probably see why). The namespace notation in APLAN is very close to that of many other languages, e.g. Python:

```python
~/work/notebooks/learnapl/contents [20] $ python
Python 3.11.3 (main, Apr 28 2025, 10:54:41) [Clang 16.0.0 (clang-1600.0.26.6)] on darwin
Type "help", "copyright", "credits" or "license" for more information.
>>> {
...   'author': 'Gödel, Kurt',
...   'title': 'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I',
...   'booktitle': 'Kurt Gödel: Collected Works, Volume I',
...   'publisher': 'Oxford University Press',
...   'year': 1931,
...   'note': 'trans. Jean van Heijenoort, publ. 1986'
... }
{'author': 'Gödel, Kurt', 'title': 'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I', 'booktitle': 'Kurt Gödel: Collected Works, Volume I', 'publisher': 'Oxford University Press', 'year': 1931, 'note': 'trans. Jean van Heijenoort, publ. 1986'}
>>>
```

Python does have a nice display form for dicts -- as they _are_ dicts, not namespaces masquerading as dicts.

We can view our namespace by converting it to JSON:

In [11]:
NSShow ← 1∘(⎕JSON⍠'Compact' 0)
NSShow ns

{
 "author": "Gödel, Kurt",
 "booktitle": "Kurt Gödel: Collected Works, Volume I",
 "note": "trans. Jean van Heijenoort, publ. 1986",
 "publisher": "Oxford University Press",
 "title": "On Formally Undecidable Propositions of Principia Mathematica and Related Systems I",
 "year": 1931
}

### Members are names, not strings

One important distinction between our namespaces and the keys in an actual dict is that Dyalog APL's namespace keys are APL _names_, not strings. This comes back to the central duality. A "key" in a namespace is really an APL variable, and so it can sometimes help to think of a namespace as an object instance. This is perhaps clearer if we look at a different way of constructing namespaces, using [⎕NS](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/ns/). 

In [12]:
ns ← ⎕NS⍬ ⍝ Construct empty anonymous namespace
ns.author ← 'Gödel, Kurt'
ns.title ← 'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I'
ns.booktitle ← 'Kurt Gödel: Collected Works, Volume I'
ns.publisher ← 'Oxford University Press'
ns.year ← 1931
ns.note ← 'trans. Jean van Heijenoort, publ. 1986'


In [13]:
NSShow ns

{
 "author": "Gödel, Kurt",
 "booktitle": "Kurt Gödel: Collected Works, Volume I",
 "note": "trans. Jean van Heijenoort, publ. 1986",
 "publisher": "Oxford University Press",
 "title": "On Formally Undecidable Propositions of Principia Mathematica and Related Systems I",
 "year": 1931
}

If we think about the namespace interpretation of the namespace (a phrase that sounds more vacuous every time I type it), it's not a big leap to see that we can make use of APL's normal assignment rules here:

In [14]:
values ← (
    'Gödel, Kurt'
    'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I'
    'Kurt Gödel: Collected Works, Volume I'
    'Oxford University Press'
    1931
    'trans. Jean van Heijenoort, publ. 1986'
)

ns ← ⎕NS⍬
ns.(author title booktitle publisher year note) ← values

NSShow ns

{
 "author": "Gödel, Kurt",
 "booktitle": "Kurt Gödel: Collected Works, Volume I",
 "note": "trans. Jean van Heijenoort, publ. 1986",
 "publisher": "Oxford University Press",
 "title": "On Formally Undecidable Propositions of Principia Mathematica and Related Systems I",
 "year": 1931
}

In Python, dict keys can be anything that's hashable, but in Dyalog APL, namespace members must have valid APL names. Dyalog jumps through hoops in order to smooth over this wrinkle. We'll see more about this when we introduce `⎕JSON` properly, but JSON, of course, assumes that keys are values:

In [19]:
NSShow ns ← 0⎕JSON'{"First name": "Kurt", "Last name": "Gödel"}'

{
 "First name": "Kurt",
 "Last name": "Gödel"
}

So far, so 100% expected. But `⎕JSON` does something under the covers, which we can uncover by actually requesting the keys:

In [20]:
]map ns

#.[JSON object]
· ~ ⍙First⍙32⍙name ⍙Last⍙32⍙name

`⎕JSON` seamlessly mangles and un-mangles keys to be valid namespace members. Thus, to access a single key, we need to do

In [22]:
ns.⍙First⍙32⍙name

Kurt

### Getting and setting values with `⎕VGET` and `⎕VSET`

Even accepting the name vs value quirk for keys, using namespaces as dicts in Dyalog used to be both clunky and slow, but in the v20 release of Dyalog APL, several core changes made it much more ergonomic and performant. We've already seen how APLAN makes it very natural to compose dict literals. Let's first show some of the problematic aspects. If we have a vector of keys and a vector of values, how can we associate those in a namespace? Prior to v20, we'd have to construct assignments as strings that were then evaluated inside the namespace:

In [ ]:
ns ← ⎕NS⍬

keys ← (
    'author'
    'title'
    'booktitle'
    'publisher'
    'year'
    'note'
)

values ← (
    'Gödel, Kurt'
    'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I'
    'Kurt Gödel: Collected Works, Volume I'
    'Oxford University Press'
    1931
    'trans. Jean van Heijenoort, publ. 1986'
)

⍝ Dyalog APL pre v20
ns {⍺.{⍎⍺,'←⍵'}¨/⍵} keys values 
⍝   │  ││    │ └───────────────── For every key-value pair
⍝   │  │└────┴─────────────────── the string '{key}←{val}' is
⍝   │  └───────────────────────── ...evaluated
⍝   └──────────────────────────── ...inside the namespace

NSShow ns

{
 "author": "Gödel, Kurt",
 "booktitle": "Kurt Gödel: Collected Works, Volume I",
 "note": "trans. Jean van Heijenoort, publ. 1986",
 "publisher": "Oxford University Press",
 "title": "On Formally Undecidable Propositions of Principia Mathematica and Related Systems I",
 "year": 1931
}

Works, sure. But if you think that is ugly, unsafe and inefficient, that's because it _is_. Fortunately, there is now a nicer, safer and faster way, using the system function [⎕VSET](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/vset/):

In [ ]:
ns ← ⎕NS⍬
ns ⎕VSET (↑keys) values  ⍝ Dyalog v20+
NSShow ns

{
 "author": "Gödel, Kurt",
 "booktitle": "Kurt Gödel: Collected Works, Volume I",
 "note": "trans. Jean van Heijenoort, publ. 1986",
 "publisher": "Oxford University Press",
 "title": "On Formally Undecidable Propositions of Principia Mathematica and Related Systems I",
 "year": 1931
}

Why the _Mix_ you say? Well, that's so that `⎕VSET` can also take a zipped list of key-value pairs:

In [26]:
ns ← ⎕NS⍬

⍝ A subset of the data...
ns ⎕VSET ('author' 'Gödel, Kurt' ⋄ 'title' 'On Formally Undecidable Propositions of Principia Mathematica and Related Systems I')
NSShow ns

{
 "author": "Gödel, Kurt",
 "title": "On Formally Undecidable Propositions of Principia Mathematica and Related Systems I"
}

The _Mix_ in the first instance is necessary in order for `⎕VSET` to be able to distinguish between the two different forms of its right argument.

Getting values as strings suffered the analogous problem to the setting which we outlined above. `⎕VSET` has a sister function, [⎕VGET](https://dyalog.github.io/documentation/21.0/language-reference-guide/system-functions/vget/) that solves that for us:

In [29]:
keys ← ('author' ⋄ 'title')

ns ⎕VGET keys

┌───────────┬───────────────────────────────────────────────────────────────────────────────────┐
│Gödel, Kurt│On Formally Undecidable Propositions of Principia Mathematica and Related Systems I│
└───────────┴───────────────────────────────────────────────────────────────────────────────────┘

`⎕VGET` has a very nice feature built in, similar to how Python's `.get()` method works on dicts -- we can specify default values to be returned if a key isn't found:

In [31]:
ns ⎕VGET ('author' '-- no author given --' ⋄ 'title' '-- no title given --' ⋄ 'year' '-- no year given --')

┌───────────┬───────────────────────────────────────────────────────────────────────────────────┬───────────────────┐
│Gödel, Kurt│On Formally Undecidable Propositions of Principia Mathematica and Related Systems I│-- no year given --│
└───────────┴───────────────────────────────────────────────────────────────────────────────────┴───────────────────┘

For a single key with default, we need to enclose the right argument:

In [32]:
ns ⎕VGET ⊂'year' '-- no year given --'

-- no year given --